In [5]:

from pathlib import Path

folder = Path.home() / "Desktop" / "POSM_CoilPCB" / "POSM_6mm_Magpylib_Verification_Package" / "field_simulation"

print("Folder exists:", folder.exists())
print("\nFiles and folders inside:")
if folder.exists():
    for item in folder.iterdir():
        print(item.name)


Folder exists: True

Files and folders inside:
plot_3d_mot_zero_backup.py
export_and_plot.py
DENSE_08mm_8Cu_6WINDING_LAYERS
plot_3d_mot_rainbow_backup.py
plot_3d_mot.py
.DS_Store
requirements.txt
POSM_6mm_3D_field_arrows.html
plot_3d_mot_backup.py
POSM_6mm_3D_field.html
use_magpylib.py
gradient_vs_current_height_3D.png
source_boards
__pycache__
README.md
height_gradient_vs_current.py.save
gradient_vs_current.png
DENSE_06mm_8Cu_7WINDING_LAYERS
DENSE_10mm_8Cu_7WINDING_LAYERS
height_gradient_vs_current.py
POSM_6mm_TRUE_3D_field_lines.html
.venv
validation.json
DENSE_12mm_8Cu_7WINDING_LAYERS
verify_magpylib_6mm.py
legacy_package_check.json
field_comparison.png


In [6]:

from pathlib import Path
import numpy as np
import magpylib as magpy
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# --------------------------------------------------
# 1. Load the 6 mm coil PCB geometry
# --------------------------------------------------
folder = Path.cwd() / "DENSE_06mm_8Cu_7WINDING_LAYERS"
vertices_file = folder / "vertices_closed_m.txt"

if not vertices_file.exists():
    raise FileNotFoundError(
        f"Could not find {vertices_file}\n"
        "Make sure JupyterLab is open in the field_simulation folder."
    )

vertices = np.loadtxt(vertices_file)
vertices = np.asarray(vertices).reshape(-1, 3)

coil = magpy.current.Polyline(
    current=1.0,
    vertices=vertices
)

print("Geometry loaded successfully.")
print(f"Number of vertices: {len(vertices)}")
print(f"Magpylib version: {magpy.__version__}")

# --------------------------------------------------
# 2. Set simulation values
# --------------------------------------------------
currents = np.linspace(0.1, 1.0, 19)  # 0.1 A to 1.0 A
heights_mm = np.linspace(0.5, 20.0, 300)
dz = 1e-5  # 10 micrometers, in meters

# --------------------------------------------------
# 3. Calculate axial magnetic field Bz in gauss
# --------------------------------------------------
def bz_gauss(height_mm, current_A):
    point = np.array([[0, 0, height_mm / 1000]])

    field = np.asarray(coil.getB(point)).reshape(-1, 3)

    # The coil object uses 1 A; scale the field to the requested current.
    return field[0, 2] * 1e4 * current_A


# --------------------------------------------------
# 4. Calculate axial gradient dBz/dz in G/cm
# --------------------------------------------------
def gradient_g_cm(height_mm, current_A):
    z = height_mm / 1000

    points = np.array([
        [0, 0, z - dz],
        [0, 0, z + dz]
    ])

    field = np.asarray(coil.getB(points)).reshape(-1, 3)
    bz = field[:, 2] * 1e4 * current_A

    # Convert the derivative from G/m to G/cm.
    return (bz[1] - bz[0]) / (2 * dz * 100)


# --------------------------------------------------
# 5. Find the axial Bz zero crossing for each current
# --------------------------------------------------
zero_heights = []

for current in currents:
    fields = [bz_gauss(h, current) for h in heights_mm]
    zero = np.nan

    for i in range(len(heights_mm) - 1):
        if fields[i] * fields[i + 1] < 0:
            zero = brentq(
                lambda h: bz_gauss(h, current),
                heights_mm[i],
                heights_mm[i + 1]
            )
            break

    zero_heights.append(zero)

# Plot 1: zero-point height versus current
plt.figure(figsize=(8, 5))
plt.plot(currents, zero_heights, "o-")
plt.xlabel("Current (A)")
plt.ylabel("Axial Bz zero-crossing height (mm)")
plt.title("Axial Zero-Crossing Height vs. Current")
plt.grid(True)
plt.tight_layout()
plt.savefig("height_vs_current.png", dpi=200)
plt.show()


# --------------------------------------------------
# 6. Calculate gradient at 6 mm for each current
# --------------------------------------------------
gradients = [
    gradient_g_cm(6.0, current)
    for current in currents
]

# Plot 2: gradient versus current
plt.figure(figsize=(8, 5))
plt.plot(currents, gradients, "o-")
plt.xlabel("Current (A)")
plt.ylabel("Axial gradient dBz/dz (G/cm)")
plt.title("Axial Gradient vs. Current at 6 mm")
plt.grid(True)
plt.tight_layout()
plt.savefig("gradient_vs_current.png", dpi=200)
plt.show()


# --------------------------------------------------
# 7. Print results
# --------------------------------------------------
print("\nSaved plots:")
print("  height_vs_current.png")
print("  gradient_vs_current.png")

print("\nCurrent (A) | Zero height (mm) | Gradient (G/cm)")
print("-" * 54)

for current, height, grad in zip(
    currents, zero_heights, gradients
):
    height_text = (
        f"{height:.4f}" if np.isfinite(height) else "No crossing"
    )
    print(f"{current:10.2f} | {height_text:16} | {grad:14.4f}")


FileNotFoundError: Could not find /Users/naimahughley/DENSE_06mm_8Cu_7WINDING_LAYERS/vertices_closed_m.txt
Make sure JupyterLab is open in the field_simulation folder.